# Final models: LR + kNN + MLP ensembles

In this notebook we compare the single models with several versions of a soft-vote ensemble, to see how each input feature affects the performance:

1. Embedding only
2. Embedding + country
3. Embedding + country + one financial feature (net margin, EBITDA margin, asset turnover or the zero-revenue flag)
4. Embedding + country + all financial ratios

For the score we use the `confidence_score` from the course skeleton, which is weighted F1 × (1 - Brier).

There are two ways to evaluate the models, and both use the deduplicated data:

- **Quick test:** one 90/10 split. It takes about 10 minutes, so we use it for trying things out.
- **5-fold cross-validation:** takes about 2 hours. We use this one for the final numbers.

The 5-fold part only runs if `RUN_FULL_CV = True` in the settings below. Training the final model only takes a few minutes.

In [1]:
import ast
import time

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

DATA_DIR = "../../data/"   # folder with df_train.pkl and df_financials_train.pkl
RUN_FULL_CV = False        # True = also run the slow 5-fold cross-validation
N_FOLDS = 5
MLP_SEEDS = 3              # random seeds per MLP in the 5-fold CV and the final model
QUICK_MLP_SEEDS = 1        # random seeds per MLP in the quick test

# 1. Loading Data

The private test set is cleaned, so we evaluate on cleaned data as well. We remove the duplicates once, before any split. If we removed them separately in train and validation, the same embedding could end up on both sides.

In [2]:
df = pd.read_pickle(DATA_DIR + "df_train.pkl")
print("rows before deduplication:", len(df))

df = df.drop_duplicates(subset="business_description_embedding").reset_index(drop=True)
print("rows after deduplication: ", len(df))

X = np.array(df["business_description_embedding"].apply(ast.literal_eval).tolist())

label_encoder = LabelEncoder().fit(df["industry"])
y = label_encoder.transform(df["industry"])
print("embedding shape:", X.shape, "| industries:", len(label_encoder.classes_))

rows before deduplication: 31706
rows after deduplication:  27128
embedding shape: (27128, 768) | industries: 25


About 4,600 rows are removed, which leaves 27,128 companies with 25 industries.

Now the financial data. 162 ids appear twice in the file, so we keep one row per company. `reindex` puts the table in the same order as `df`. Companies without financial data get an empty row, which is handled as missing later on.

In [3]:
df_fin = pd.read_pickle(DATA_DIR + "df_financials_train.pkl").groupby("id").first()
fin = df_fin.reindex(df["id"].values).reset_index(drop=True)
print("companies without financial data:", fin["country_code"].isna().sum())

companies without financial data: 2206


# 2. Scoring function

This is the `confidence_score` from the course skeleton. It can never be higher than the weighted F1, because (1 - Brier) is at most 1.

In [4]:
def confidence_score(y_true, y_pred, confidence):
    f1 = f1_score(y_true, y_pred, average="weighted")
    correct = (y_true == y_pred).astype(float)
    brier = np.mean((confidence - correct) ** 2)
    return f1 * (1 - brier)


def score_from_probabilities(y_true, P):
    """Prediction = most likely class. Confidence = its probability."""
    return confidence_score(y_true, P.argmax(axis=1), P.max(axis=1))

# 3. Extra features: country and financial ratios

Each input can be switched on or off, so that we can test them separately. `ExtraFeatures` gets a list with these groups:

- `country`: the one-hot country columns.
- `net_profit_margin`, `ebitda_margin`, `asset_turnover`: the scaled ratio, a missing flag and a zero-revenue flag for this ratio.
- `zero_revenue`: only one flag for zero revenue, without any ratio.
- `no_row`: one flag for companies without financial data.

All statistics (percentiles, medians, scaling, country categories) are learned in `fit` from the training rows only and then reused in `transform`. If we calculated them on all rows, information from the test rows would leak into the training.

The ratios are handled the same way as in `03_financial_analysis.ipynb`:

- An `inf` margin or an asset turnover of 0 means zero revenue, so the company gets a zero-revenue flag.
- A negative asset turnover is treated as missing.
- Missing values get the median plus a missing flag. EBITDA alone is missing for about 65% of the companies.
- Extreme outliers are compressed with a signed log and then clipped at the 1st and 99th percentile of the training rows.

Country becomes one-hot columns, and countries with fewer than 20 companies are grouped together.

In [5]:
RATIOS = ["net_profit_margin", "ebitda_margin", "asset_turnover"]


class ExtraFeatures:
    def __init__(self, groups):
        self.groups = groups

    def _country(self, fin):
        return fin["country_code"].fillna("unknown").to_numpy().reshape(-1, 1)

    def _clean_ratio(self, fin, ratio):
        v = fin[ratio].to_numpy(dtype=float).copy()
        zero_revenue = (v == 0) if ratio == "asset_turnover" else np.isinf(v)
        if ratio == "asset_turnover":
            v[v < 0] = np.nan
        v[np.isinf(v)] = np.nan
        missing = np.isnan(v)
        signed_log = np.sign(v) * np.log1p(np.abs(v))
        return signed_log, missing, zero_revenue

    def _zero_revenue(self, fin):
        net_margin = fin["net_profit_margin"].to_numpy(dtype=float)
        turnover = fin["asset_turnover"].to_numpy(dtype=float)
        return np.isinf(net_margin) | (turnover == 0)

    def fit(self, fin):
        if "country" in self.groups:
            self.encoder = OneHotEncoder(min_frequency=20, handle_unknown="infrequent_if_exist",
                                         sparse_output=False).fit(self._country(fin))
        self.stats = {}
        for ratio in RATIOS:
            if ratio in self.groups:
                s, missing, _ = self._clean_ratio(fin, ratio)
                low, high = np.nanquantile(s, [0.01, 0.99])
                s = np.clip(s, low, high)
                median = np.nanmedian(s)
                s[missing] = median
                self.stats[ratio] = (low, high, median, s.mean(), s.std())
        return self

    def transform(self, fin):
        columns = []
        for group in self.groups:
            if group == "country":
                columns.append(self.encoder.transform(self._country(fin)))
            elif group == "no_row":
                columns.append(fin["country_code"].isna().to_numpy(dtype=float).reshape(-1, 1))
            elif group == "zero_revenue":
                columns.append(self._zero_revenue(fin).astype(float).reshape(-1, 1))
            else:
                low, high, median, mean, std = self.stats[group]
                s, missing, zero_revenue = self._clean_ratio(fin, group)
                s = np.clip(s, low, high)
                s[missing] = median
                columns.append(np.column_stack([(s - mean) / std, missing, zero_revenue]).astype(float))
        return np.hstack(columns)

# 4. The models

We use three models that work quite differently, so they hopefully make different mistakes:

- **Logistic regression** is linear and captures the broad directions in the embedding space.
- **kNN** looks at the most similar companies, so it can pick up small local clusters.
- **MLP** is non-linear and can combine features in ways a straight boundary can't.

kNN always uses only the embedding, because extra 0/1 columns would distort the cosine distance between the descriptions. Its embeddings are centered and normalized again, so that the distance focuses on what differs between companies.

In [6]:
def predict_lr(X_train, y_train, X_test, C=3):
    return LogisticRegression(C=C, max_iter=3000).fit(X_train, y_train).predict_proba(X_test)


def predict_knn(X_train, y_train, X_test):
    mean_embedding = X_train.mean(axis=0)

    def center(Z):
        Z = Z - mean_embedding
        return Z / np.linalg.norm(Z, axis=1, keepdims=True)

    knn = KNeighborsClassifier(n_neighbors=15, metric="cosine", weights=lambda d: np.exp(-10 * d))
    return knn.fit(center(X_train), y_train).predict_proba(center(X_test))


def predict_mlp(X_train, y_train, X_test, seeds, progress, label):
    P = 0
    for seed in range(seeds):
        mlp = make_pipeline(
            StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(512,), alpha=1e-2, early_stopping=True, random_state=seed),
        )
        P = P + mlp.fit(X_train, y_train).predict_proba(X_test) / seeds
        progress.step(f"{label}: MLP seed {seed + 1}/{seeds}")
    return P


def soft_vote(*probabilities):
    return sum(probabilities) / len(probabilities)

# 5. Shared experiment code

`run_all_models` trains every model on one set of training rows and predicts the test rows. The quick test calls it once and the 5-fold cross-validation calls it five times.

`Progress` prints a line after each finished step, with the percentage done and a rough guess of the time left.

In [7]:
FEATURE_SETS = {
    "embedding only": [],
    "embedding + country": ["country"],
    "embedding + country + net margin": ["country", "net_profit_margin"],
    "embedding + country + EBITDA margin": ["country", "ebitda_margin"],
    "embedding + country + asset turnover": ["country", "asset_turnover"],
    "embedding + country + zero revenue": ["country", "zero_revenue"],
    "embedding + country + financials": ["country", "no_row", "net_profit_margin", "ebitda_margin", "asset_turnover"],
}


class Progress:
    def __init__(self, total_steps):
        self.total = total_steps
        self.done = 0
        self.start = time.time()

    def step(self, text):
        self.done += 1
        elapsed = (time.time() - self.start) / 60
        remaining = elapsed / self.done * (self.total - self.done)
        print(f"[{self.done}/{self.total} | {self.done / self.total:.0%}] {text} | "
              f"elapsed {elapsed:.1f} min, about {remaining:.1f} min left", flush=True)


def steps_per_run(seeds):
    return 2 + len(FEATURE_SETS) * (1 + seeds)


def run_all_models(train_idx, test_idx, seeds, progress, label):
    X_train, y_train, X_test = X[train_idx], y[train_idx], X[test_idx]
    probabilities = {}

    probabilities["LR C=1 (baseline)"] = predict_lr(X_train, y_train, X_test, C=1)
    progress.step(f"{label}: baseline LR")
    P_knn = predict_knn(X_train, y_train, X_test)
    probabilities["kNN"] = P_knn
    progress.step(f"{label}: kNN")

    for name, groups in FEATURE_SETS.items():
        if not groups:
            F_train, F_test = X_train, X_test
        else:
            extra = ExtraFeatures(groups).fit(fin.iloc[train_idx])
            F_train = np.hstack([X_train, extra.transform(fin.iloc[train_idx])])
            F_test = np.hstack([X_test, extra.transform(fin.iloc[test_idx])])

        P_lr = predict_lr(F_train, y_train, F_test)
        progress.step(f"{label}: {name}: LR")
        P_mlp = predict_mlp(F_train, y_train, F_test, seeds, progress, f"{label}: {name}")

        probabilities[f"LR C=3 | {name}"] = P_lr
        probabilities[f"MLP | {name}"] = P_mlp
        probabilities[f"Ensemble | {name}"] = soft_vote(P_lr, P_knn, P_mlp)

    return probabilities

# 6. Quick test: one 90/10 split

Each model is trained once and tested on 10% of the companies. We use this for experiments, but there are two limits:

- The score of one split moves by about +/-0.01 depending on which companies land in the test part, so we shouldn't trust small differences here.
- The MLP uses fewer seeds here to save time, so its scores can be a bit lower than in the 5-fold run.

In [8]:
train_idx, test_idx = train_test_split(np.arange(len(y)), test_size=0.1, random_state=42, stratify=y)
y_quick = y[test_idx]

progress = Progress(steps_per_run(QUICK_MLP_SEEDS))
quick = run_all_models(train_idx, test_idx, QUICK_MLP_SEEDS, progress, "quick test")

[1/8 | 12%] quick test: baseline LR | elapsed 0.1 min, about 0.6 min left
[2/8 | 25%] quick test: kNN | elapsed 0.1 min, about 0.4 min left
[3/8 | 38%] quick test: embedding only: LR | elapsed 0.3 min, about 0.5 min left
[4/8 | 50%] quick test: embedding only: MLP seed 1/1 | elapsed 1.1 min, about 1.1 min left
[5/8 | 62%] quick test: embedding + country: LR | elapsed 1.3 min, about 0.8 min left
[6/8 | 75%] quick test: embedding + country: MLP seed 1/1 | elapsed 2.5 min, about 0.8 min left
[7/8 | 88%] quick test: embedding + country + financials: LR | elapsed 2.7 min, about 0.4 min left
[8/8 | 100%] quick test: embedding + country + financials: MLP seed 1/1 | elapsed 3.8 min, about 0.0 min left


In [9]:
quick_results = pd.DataFrame([
    {"model": name, "F1": f1_score(y_quick, P.argmax(axis=1), average="weighted"),
     "score": score_from_probabilities(y_quick, P)}
    for name, P in quick.items()
]).set_index("model")
quick_results["gain vs baseline"] = quick_results["score"] - quick_results.loc["LR C=1 (baseline)", "score"]
quick_results.sort_values("score", ascending=False).round(4)

,F1,score,gain vs baseline
model,,,
Ensemble | embedding + country,0.7615,0.6521,0.0369
Ensemble | embedding + country + financials,0.7602,0.6516,0.0364
Ensemble | embedding only,0.7530,0.6423,0.0270
MLP | embedding + country + financials,0.7451,0.6300,0.0147
LR C=3 | embedding + country + financials,0.7426,0.6294,0.0141
LR C=3 | embedding + country,0.7425,0.6284,0.0132
kNN,0.7422,0.6278,0.0126
LR C=3 | embedding only,0.7410,0.6262,0.0110
MLP | embedding + country,0.7377,0.6222,0.0069


(These numbers are from the run with only three feature sets. To update after the next run.)

All three ensembles are better than every single model, so combining them helps. The best one scores 0.6521, compared to 0.6153 for the logistic regression baseline.

Between the ensembles we can't really say much from this table. The difference between embedding + country (0.6521) and embedding only (0.6423) is about the size of the noise of one split, so we need the 5-fold run to decide that.

# 7. Full 5-fold cross-validation (final numbers)

Set `RUN_FULL_CV = True` in the settings to run this part. Every company is then tested exactly once, by models that never saw it.

The progress lines show how far along the run is. The time left is only a rough guess, especially during the first steps.

In [11]:
oof = {}
fold_scores = []

if RUN_FULL_CV:
    folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=0).split(X, y))
    progress = Progress(N_FOLDS * steps_per_run(MLP_SEEDS))

    for fold, (train_idx, test_idx) in enumerate(folds):
        probabilities = run_all_models(train_idx, test_idx, MLP_SEEDS, progress, f"fold {fold + 1}/{N_FOLDS}")
        for name, P in probabilities.items():
            oof.setdefault(name, np.zeros((len(y), P.shape[1])))[test_idx] = P
            fold_scores.append({"fold": fold, "model": name,
                                "F1": f1_score(y[test_idx], P.argmax(axis=1), average="weighted"),
                                "score": score_from_probabilities(y[test_idx], P)})
else:
    print("5-fold cross-validation skipped. Set RUN_FULL_CV = True to run it.")

5-fold cross-validation skipped. Set RUN_FULL_CV = True to run it.


### 5-fold results

`score_std` shows how much the score varies across the folds. Differences below about 0.005 between two rows could just be noise, unless the comparison below shows that they are consistent.

In [12]:
if RUN_FULL_CV:
    results = pd.DataFrame(fold_scores)
    summary = results.groupby("model", sort=False).agg(
        F1=("F1", "mean"), score=("score", "mean"), score_std=("score", "std")
    ).round(4)
    display(summary.sort_values("score", ascending=False))
else:
    print("Skipped: needs RUN_FULL_CV = True.")

Skipped: needs RUN_FULL_CV = True.


### Do the extra features really help?

The folds are the same for all models, so we can compare the ensembles fold by fold. This removes the differences between the folds, so it is much more sensitive than just comparing two averages.

We compare with the embedding-only ensemble to see what all extras together add. All financial features are added on top of country, so the gain against embedding + country shows what one financial feature adds by itself.

In [13]:
if RUN_FULL_CV:
    pivot = results.pivot(index="fold", columns="model", values="score")
    base = pivot["Ensemble | embedding only"]
    country = pivot["Ensemble | embedding + country"]
    baseline = pivot["LR C=1 (baseline)"]

    display(pd.DataFrame([
        {"ensemble": name,
         "score": pivot[f"Ensemble | {name}"].mean(),
         "gain vs embedding only": (pivot[f"Ensemble | {name}"] - base).mean(),
         "gain std": (pivot[f"Ensemble | {name}"] - base).std(),
         "gain vs country": (pivot[f"Ensemble | {name}"] - country).mean(),
         "gain vs country std": (pivot[f"Ensemble | {name}"] - country).std(),
         "folds better than country": int(((pivot[f"Ensemble | {name}"] - country) > 0).sum()),
         "gain vs baseline LR": (pivot[f"Ensemble | {name}"] - baseline).mean(),
         "folds improved": int(((pivot[f"Ensemble | {name}"] - base) > 0).sum())}
        for name in FEATURE_SETS
    ]).round(4))
else:
    print("Skipped: needs RUN_FULL_CV = True.")

Skipped: needs RUN_FULL_CV = True.


# 8. Uncertainty: does the confidence mean what it says?

If the confidence is well calibrated, the accuracy in each bin should be about the same as the confidence. For example, if the model says 80%, it should be right about 80% of the time. We can then flag the least confident 20% of the predictions for human review.

We do this with the embedding-only ensemble, using the 5-fold predictions if they exist and the quick test otherwise. If the final model uses a different feature set, this should be changed to match.

In [14]:
if RUN_FULL_CV:
    P, y_true = oof["Ensemble | embedding only"], y
else:
    P, y_true = quick["Ensemble | embedding only"], y_quick

y_pred = P.argmax(axis=1)
confidence = P.max(axis=1)
correct = y_pred == y_true

bins = pd.cut(confidence, [0, 0.4, 0.6, 0.8, 0.9, 1.0])
pd.DataFrame({"confidence bin": bins, "correct": correct, "confidence": confidence}).groupby(
    "confidence bin", observed=True
).agg(share=("correct", lambda c: len(c) / len(y_true)), mean_confidence=("confidence", "mean"),
      accuracy=("correct", "mean")).round(3)

,share,mean_confidence,accuracy
confidence bin,,,
"(0.0, 0.4]",0.078,0.329,0.335
"(0.4, 0.6]",0.209,0.510,0.555
"(0.6, 0.8]",0.245,0.704,0.761
"(0.8, 0.9]",0.171,0.853,0.864
"(0.9, 1.0]",0.296,0.955,0.947


In the quick test the confidence matches the accuracy quite well. In the highest bin the model says 95.5% and is right 94.7% of the time. In the middle bins it is a bit underconfident, for example 70.4% confidence with 76.1% accuracy.

In [15]:
flagged = confidence < np.quantile(confidence, 0.2)
print(f"accuracy on all companies:      {correct.mean():.3f}")
print(f"accuracy on flagged 20%:        {correct[flagged].mean():.3f}")
print(f"accuracy on the remaining 80%:  {correct[~flagged].mean():.3f}")

accuracy on all companies:      0.757
accuracy on flagged 20%:        0.427
accuracy on the remaining 80%:  0.840


Flagging works well in the quick test. The flagged 20% only have an accuracy of 42.7%, while the remaining 80% reach 84.0%. So the confidence really does find the predictions that are likely to be wrong.

# 9. Final model for submission

The final model trains on all deduplicated rows, since the cross-validation already gave us the performance estimate. `predict` returns `(y_pred, y_confidence)`, with the confidence between 0 and 1. `y_pred` contains the encoded labels, and `label_encoder.inverse_transform(y_pred)` gives the industry names.

The feature set is chosen here with `FINAL_FEATURES`. The private test set comes with financial data, so we can use the extra features in the final model. Which ones we use should come from the 5-fold results.

In [16]:
FINAL_FEATURES = "embedding only"   # one of the names in FEATURE_SETS


def train_final_model(X, y, fin, groups):
    extra = ExtraFeatures(groups).fit(fin) if groups else None
    F = X if extra is None else np.hstack([X, extra.transform(fin)])

    lr = LogisticRegression(C=3, max_iter=3000).fit(F, y)
    mlps = [make_pipeline(StandardScaler(),
                          MLPClassifier(hidden_layer_sizes=(512,), alpha=1e-2, early_stopping=True, random_state=seed)
                          ).fit(F, y) for seed in range(MLP_SEEDS)]

    mean_embedding = X.mean(axis=0)

    def center(Z):
        Z = Z - mean_embedding
        return Z / np.linalg.norm(Z, axis=1, keepdims=True)

    knn = KNeighborsClassifier(n_neighbors=15, metric="cosine", weights=lambda d: np.exp(-10 * d))
    knn.fit(center(X), y)

    def predict(X_new, fin_new=None):
        F_new = X_new if extra is None else np.hstack([X_new, extra.transform(fin_new)])
        P_mlp = sum(m.predict_proba(F_new) for m in mlps) / len(mlps)
        P = soft_vote(lr.predict_proba(F_new), knn.predict_proba(center(X_new)), P_mlp)
        return P.argmax(axis=1), P.max(axis=1)

    return predict


start = time.time()
predict = train_final_model(X, y, fin, FEATURE_SETS[FINAL_FEATURES])
print(f"training time: {(time.time() - start) / 60:.1f} min (limit: 45 min)")

training time: 2.9 min (limit: 45 min)


A quick check of the output format on five training rows. This says nothing about performance, because the model has already seen these rows. On the test set the call is `predict(X_test)`, or `predict(X_test, fin_test)` if the model uses extra features. `fin_test` has to be prepared the same way as `fin`.

In [17]:
y_pred_check, y_confidence_check = predict(X[:5], fin.iloc[:5])
assert np.all((y_confidence_check >= 0) & (y_confidence_check <= 1))
print(label_encoder.inverse_transform(y_pred_check))
print(y_confidence_check.round(3))

['Banks' 'Materials' 'Commercial & Professional Services' 'Insurance'
 'Capital Goods']
[0.961 0.849 0.477 0.993 0.819]
